In [ ]:
from pathlib import Path
import numpy as np 
import tifffile as tf
import pandas as pd
import skimage
from sklearn.preprocessing import MinMaxScaler
import pickle
import scipy.cluster.hierarchy as sch 
import matplotlib.pyplot as plt
from collections import Counter
import cv2
from matplotlib_scalebar.scalebar import ScaleBar
from pyFlowSOM import map_data_to_nodes, som
from alpineer import io_utils, misc_utils
from pyarrow.lib import ArrowInvalid


In [ ]:
key  = "" # sample identififer

In [ ]:
registered_img_folder_path = Path("") # path to registered images

In [ ]:
cell_clustering_path = r"" # path to cell phenotyping results

In [ ]:
pixel_clustering_path = r"" # path to pixel clustering results

In [ ]:
all_IF_channels = ['PODXL', 'LRP2', 'PANCK', 'CD20', 'CD68']

In [ ]:
IF_channel_list =  ['PODXL', 'LRP2', 'PANCK', 'CD20', 'CD68']

In [ ]:
all_color_channels = ['HE_R', 'HE_G', 'HE_B', 'PAS_R', 'PAS_G', 'PAS_B', 'T_R', 'T_G', 'T_B']

In [ ]:
color_channel_list = ['HE_R', 'HE_G', 'HE_B', 'PAS_R', 'PAS_G', 'PAS_B', 'T_R', 'T_G', 'T_B']

In [ ]:
all_channels =  ['PODXL', 'LRP2', 'PANCK', 'CD20', 'CD68', 'HE_R', 'HE_G', 'HE_B', 'PAS_R', 'PAS_G', 'PAS_B', 'T_R', 'T_G', 'T_B']

In [ ]:
threshold_exists = True  
suffix = ''

In [ ]:
import distinctipy
def create_color_dict(labels_unique):
    color_arr = distinctipy.get_colors(len(labels_unique))
    cell_type_color_dict = {}
    for i in range(len(labels_unique)):
        cell_type_color_dict[labels_unique[i]] = color_arr[i]
    return cell_type_color_dict

In [ ]:
if(threshold_exists):
    with open(pixel_clustering_path +  '/thr_dict' + suffix + '.pickle', 'rb') as f: 
            thr_dict = pickle.load(f)

In [ ]:
def threshold_image(img, channel):
    if(channel == 'DAPI'):
        _, mask = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    else:
        _, mask = cv2.threshold(img, thr_dict[channel], 255, cv2.THRESH_BINARY)
    img[mask==0] = 0

    return img 

In [ ]:
def construct_pixels_df(registered_img_folder_path, all_IF_channels, IF_channels_list, all_color_channels, color_channels_list):
    non_dapi_IF_images_paths = [f for f in registered_img_folder_path.glob('*tif') if 'CH1' not in str(f) and 'registered' in str(f) and 'Cycle' in str(f)]
    non_dapi_images = [tf.imread(f) for f in non_dapi_IF_images_paths]
    selected_non_dapi_images = []
    


    IF_channel_idx = []
    for channel in IF_channels_list:
        IF_channel_idx.append(all_IF_channels.index(channel))
    
    color_channel_idx = []
    for channel in color_channels_list:
        color_channel_idx.append(all_color_channels.index(channel))

    for i in range(len(IF_channel_idx)):
        print(IF_channels_list[i])
        print(all_IF_channels[IF_channel_idx[i]])
        img = non_dapi_images[IF_channel_idx[i]]
        p2, p98 = np.percentile(img, (0, 99.9))
        img_rescale = skimage.exposure.rescale_intensity(img, in_range=(p2, p98), out_range=(0, 255))
        img_rescale_thr = threshold_image(img_rescale, IF_channels_list[i])
        selected_non_dapi_images.append(img_rescale_thr)
    
    non_dapi_stack = np.stack(selected_non_dapi_images)

    num_images, height, width = non_dapi_stack.shape
    num_pixels = height * width
    non_dapi_stack_flattened = non_dapi_stack.reshape(num_images, num_pixels).T
    all_pixels_IF_df = pd.DataFrame(non_dapi_stack_flattened, columns=IF_channels_list)


    color_images_paths = [f for f in registered_img_folder_path.glob('*tif') if 'Cycle' not in str(f)]
    color_images_rgb_dict = {}
    for path in color_images_paths:

        stain = str(path).split('\\')[-1].split('_')[0]
        stain_img = tf.imread(path)
        black_pixels_mask = np.all(stain_img == [0, 0, 0], axis=2)
        stain_img[black_pixels_mask] = [np.max(stain_img), np.max(stain_img), np.max(stain_img)]
        color_images_rgb_dict[stain] = stain_img
    
    color_stack_rgb = np.concatenate(list(color_images_rgb_dict.values()), axis=-1)

    for i in range(len(color_channel_idx)):
        img = color_stack_rgb[:, :, color_channel_idx[i]]
        p2, p98 = np.percentile(img, (0, 99.9))
        img_rescale = skimage.exposure.rescale_intensity(img, in_range=(p2, p98), out_range=(0, 255))
        color_stack_rgb[:, :, color_channel_idx[i]] = img_rescale

    height, width, num_images = color_stack_rgb.shape
    num_pixels = height * width

    color_stack_rgb_flattened = color_stack_rgb.reshape(num_pixels, num_images)

    all_pixels_color_df = pd.DataFrame(color_stack_rgb_flattened, columns=color_channel_list)

    all_pixels_df = all_pixels_IF_df.join(all_pixels_color_df)
    x = color_stack_rgb.shape[0]
    y = color_stack_rgb.shape[1]
        
    pixel_range = [i for i in range(x)]
    all_pixels_df['X'] = np.repeat(pixel_range, y)
    all_pixels_df['Y'] = [i for i in range(y)] * x

    return all_pixels_df, all_pixels_color_df, all_pixels_IF_df, non_dapi_stack, color_stack_rgb

In [ ]:
def compute_som_weights(pixels_subset_no_meta, prefix, xdim = 10, ydim = 10, num_passes = 1, lr_start=0.05, lr_end=0.01, seed = 42):
    som_weights = som(
            data=pixels_subset_no_meta.values.astype(np.float64), xdim=xdim, ydim=ydim,
            rlen=num_passes, alpha_range=(lr_start, lr_end), seed=seed
        )

    som_weights = np.reshape(som_weights, (xdim * ydim, som_weights.shape[-1]))
    weights_df = pd.DataFrame(som_weights, columns=pixels_subset_no_meta.columns.values)
    return weights_df

In [ ]:
def assign_som_clusters(weights_df, pixels_no_meta, batch_size): 
    cluster_labels = []
    
    for i in np.arange(0, pixels_no_meta.shape[0], batch_size):

        cluster_labels.append(map_data_to_nodes(
            weights_df.values.astype(np.float64),
            pixels_no_meta.loc[
                i:min(i + batch_size-1, pixels_no_meta.shape[0]), pixels_no_meta.columns.values
            ].values.astype(np.float64)
        )[0])
    print(np.concatenate(cluster_labels).shape)
    pixels_no_meta['pixel_som_cluster'] = np.concatenate(cluster_labels)
    return pixels_no_meta
    
    

In [ ]:
def cluster(pixels_subset, pixels, prefix, batch_size, drop_position, weights_df = None, xdim = None, ydim= None):
    if(drop_position):
        pixels_subset_no_meta = pixels_subset.drop(columns=['X', 'Y'])
        pixels_no_meta = pixels.drop(columns=['X', 'Y'])
    else:
        pixels_subset_no_meta = pixels_subset
        pixels_no_meta = pixels
    print('computing som weights')
    if(weights_df is None):
        if(xdim is None and ydim is None):
            weights_df = compute_som_weights(pixels_subset_no_meta, prefix)
        else:
            weights_df = compute_som_weights(pixels_subset_no_meta, prefix, xdim=xdim, ydim=ydim)
        print(weights_df.shape)
    print('assigning som clusters')
    pixels_no_meta_som_clusters = assign_som_clusters(weights_df, pixels_no_meta, batch_size)
   

    return pixels_no_meta_som_clusters, weights_df

In [ ]:
def create_clustered_pixel(x, y, labels):
    x_max = np.max(x)
    y_max = np.max(y)
    img_labeled = np.zeros((x_max+1, y_max+1)) * np.nan
    img_labeled[x, y] = labels
    return img_labeled 

In [ ]:
all_pixels_df, all_pixels_color_df, all_pixels_IF_df, IF_stack, color_stack = construct_pixels_df(registered_img_folder_path, all_IF_channels, IF_channel_list, all_color_channels, color_channel_list)

In [ ]:
with open('/'.join(str(registered_img_folder_path).split('\\')) + '/roi_coord_dict.pickle', 'rb') as f:
    roi = pickle.load(f)

In [ ]:
Path.mkdir(Path('/'.join(pixel_clustering_path.split('/')[:-1])), exist_ok=True)

In [ ]:
Path.mkdir(Path(pixel_clustering_path), exist_ok=True)

In [ ]:
with open(pixel_clustering_path + '/IF_stack.pickle', 'wb') as f: 
    pickle.dump(IF_stack, f)

In [ ]:
with open(pixel_clustering_path + '/color_stack_rgb.pickle', 'wb') as f: 
    pickle.dump(color_stack, f)

In [ ]:
all_pixels_df.to_csv(pixel_clustering_path + '/pixels.csv')

In [ ]:
non_empty_pixels = all_pixels_df[~(((all_pixels_df.iloc[:,0].le(0.1*255))) & ((all_pixels_df.iloc[:,len(IF_channel_list):-3].ge(0.9*255)).all(axis=1)))]


In [ ]:
with open(cell_clustering_path + '/segmentation_mask_merged_within_contour_for_pixel_clustering.pickle', 'rb') as f:
    cell_segmentation_mask = pickle.load(f)

In [ ]:
x = non_empty_pixels['X']
y = non_empty_pixels['Y']
non_empty_pixels_no_meta = non_empty_pixels.drop(columns=['X', 'Y'])
scaler = MinMaxScaler()
x_scaled = scaler.fit_transform(non_empty_pixels_no_meta)
x_scaled = scaler.transform(non_empty_pixels_no_meta)
non_empty_pixels_scaled = pd.DataFrame(x_scaled, columns=non_empty_pixels_no_meta.columns)

non_empty_pixels_scaled['X'] = list(x)
non_empty_pixels_scaled['Y'] = list(y)

In [ ]:
non_empty_pixels.to_csv(pixel_clustering_path + '/non_empty_pixels.csv')

In [ ]:
non_empty_pixels_scaled.to_csv(pixel_clustering_path + '/non_empty_pixels_scaled_weights_transferred.csv')

In [ ]:
non_empty_pixels_som_clusters_no_meta, weights_df =  cluster(non_empty_pixels_scaled, non_empty_pixels_scaled, 'weights', 100000000, drop_position=True, weights_df=None, xdim=4, ydim=4)

In [ ]:
non_empty_pixels_som_clusters = non_empty_pixels_som_clusters_no_meta.copy()

In [ ]:
non_empty_pixels_som_clusters['X'] = list(non_empty_pixels_scaled['X'])

In [ ]:
non_empty_pixels_som_clusters['Y'] = list(non_empty_pixels_scaled['Y'])

In [ ]:
non_empty_pixels_som_clusters.to_csv(pixel_clustering_path + '/non_empty_pixels_som_clusters_weights_transferred.csv')

In [ ]:
weights_df.to_csv(pixel_clustering_path + '/som_weights.csv')

In [ ]:
with open(str(registered_img_folder_path) + '/IF_scalebar_dict.pickle', 'rb') as f:
    scalebar_dict = pickle.load(f)

In [ ]:
cell_segmentation_mask_binary = np.zeros(cell_segmentation_mask.shape)
cell_segmentation_mask_binary[cell_segmentation_mask!=0] = 1
cell_segmentation_mask_binary = cv2.dilate(cv2.erode(cell_segmentation_mask_binary, np.ones((3, 3), np.uint8), iterations=1), np.ones((83, 83), np.uint8), iterations=1)

contours, _ = cv2.findContours(cell_segmentation_mask_binary.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contour = max(contours, key=cv2.contourArea)

contour_drawn = np.zeros((cell_segmentation_mask_binary.shape[0], cell_segmentation_mask_binary.shape[1]))
contour_drawn = cv2.drawContours(contour_drawn, [contour], -1, (255, 255, 255), -1)
contour_drawn = contour_drawn

In [ ]:
x = non_empty_pixels_som_clusters.X.tolist()
y = non_empty_pixels_som_clusters.Y.tolist()
wsi_labeled = create_clustered_pixel(x, y, np.array(list(non_empty_pixels_som_clusters.pixel_som_cluster)))

wsi_labeled = np.pad(wsi_labeled, ((0, int(np.ceil(wsi_labeled.shape[0]/512)*512 - wsi_labeled.shape[0])), (0, int(np.ceil(wsi_labeled.shape[1]/512)*512 - wsi_labeled.shape[1]))))
wsi_labeled = np.pad(wsi_labeled, ((0, int(cell_segmentation_mask.shape[0]- wsi_labeled.shape[0])), (0, int(cell_segmentation_mask.shape[1]- wsi_labeled.shape[1]))))
wsi_labeled[wsi_labeled == 0] = np.nan
wsi_labeled[contour_drawn == 0] = np.nan

In [ ]:
with open(pixel_clustering_path + '/wsi_labeled_weights_transferred.pickle', 'wb') as f:
    pickle.dump(wsi_labeled, f)

In [ ]:
cluster_color_dict = create_color_dict(list(set(list(non_empty_pixels_som_clusters['pixel_som_cluster'].unique()))))
with open(pixel_clustering_path + '/cluster_color_dict.pickle', 'wb') as f:
    pickle.dump(cluster_color_dict, f)

In [ ]:
fig, ax = plt.subplots(1)
fig.set_figheight(150)
fig.set_figwidth(150)

for name, coord in roi.items():
    wsi_labeled = cv2.rectangle(wsi_labeled, (coord[2], coord[0]), (coord[3], coord[1]), np.nan, 50)
cmap = plt.cm.colors.ListedColormap(list(cluster_color_dict.values()))
ax.imshow(wsi_labeled, cmap=cmap)
ax.axis('off')

In [ ]:
factor_downsample = 4

In [ ]:
fig, ax = plt.subplots(1, len(list(roi.keys())))
fig.set_figheight(30)
fig.set_figwidth(50)
i = 0
j = 0
for name, coord in roi.items():
    print(name)
    img_labeled = wsi_labeled[coord[0]:coord[1], coord[2]:coord[3]]
    
    roi_scalebar = ScaleBar(0.1771*factor_downsample, 'um', length_fraction=0.25, font_properties={'size':50})
    ax[j].add_artist(roi_scalebar)
    ax[j].imshow(img_labeled, cmap=plt.cm.colors.ListedColormap(list(cluster_color_dict.values())), vmin=1, vmax=16)
    ax[j].axis('off')
    j+=1 

    plt.subplots_adjust(wspace=0.025, hspace=0.025)

In [ ]:
import matplotlib.patches as mpatches
fig, ax = plt.subplots(1)

i = 0
patches = [
    mpatches.Patch(color=color, label=label)
    for label, color in zip(list(cluster_color_dict.keys()), list(cluster_color_dict.values()))]
ax.legend(patches, list(cluster_color_dict.keys()), loc='center', frameon=False, ncol=int(len(list(cluster_color_dict.keys()))))
ax.axis('off')
i+=1
plt.show()